In [402]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import calliope

# We increase logging verbosity
calliope.set_log_verbosity("INFO", include_solver_output=True)

model = calliope.read_yaml("model.yaml")

[2025-11-10 11:39:54] INFO     Math init | loading pre-defined math.
[2025-11-10 11:39:54] INFO     Math init | loading math files {'storage_inter_cluster', 'spores', 'base', 'operate', 'milp'}.
[2025-11-10 11:39:54] INFO     Model: preprocessing data
[2025-11-10 11:39:54] INFO     Math build | building applied math with ['base'].
[2025-11-10 11:39:54] INFO     input data `color` not defined in model math; it will not be available in the optimisation problem.
[2025-11-10 11:39:54] INFO     input data `name` not defined in model math; it will not be available in the optimisation problem.
[2025-11-10 11:39:54] INFO     input data `link_to` not defined in model math; it will not be available in the optimisation problem.
[2025-11-10 11:39:54] INFO     input data `link_from` not defined in model math; it will not be available in the optimisation problem.
[2025-11-10 11:39:54] WARNING  ModelWarning: Only one timestep defined. Inferring timestep resolution to be 1 hour

[2025-11-10 11:39:54] 

In [403]:
model.inputs

<xarray.Dataset> Size: 4kB
Dimensions:                     (costs: 1, techs: 13, nodes: 8, carriers: 2,
                                 timesteps: 1)
Coordinates:
  * costs                       (costs) object 8B 'monetary'
  * techs                       (techs) object 104B 'SE1_to_TE2' ... 'supply_...
  * carriers                    (carriers) object 16B 'electricity' 'heat'
  * nodes                       (nodes) object 64B 'D1' 'D2' ... 'TH1' 'TH2'
  * timesteps                   (timesteps) datetime64[ns] 8B 2050-01-01
Data variables: (12/23)
    bigM                        float64 8B 1.0
    cost_interest_rate          (costs) float64 8B 1.0
    objective_cost_weights      (costs) float64 8B 1.0
    base_tech                   (techs) object 104B 'transmission' ... 'supply'
    carrier_in                  (nodes, techs, carriers) bool 208B False ... ...
    color                       (techs) object 104B '#6783E3' ... '#C98AAD'
    ...                          ...
    link_from                   (techs) object 104B 'TE2' 'TE1' ... nan nan
    cost_flow_cap_per_distance  (costs, techs) float64 104B 50.0 50.0 ... nan
    definition_matrix           (nodes, techs, carriers) bool 208B False ... ...
    distance                    (techs) float64 104B 0.1308 0.6853 ... nan nan
    timestep_resolution         (timesteps) float64 8B 1.0
    timestep_weights            (timesteps) float64 8B 1.0

In [404]:
model.inputs.flow_cap_max.to_series().dropna()

techs
SE1_to_TE2            2000.0
SH1_to_TE1            2000.0
SH1_to_TH1            2000.0
TE1_to_D1             2000.0
TE1_to_TE2            2000.0
TE2_to_D2             2000.0
TH1_to_D1             2000.0
TH1_to_TH2            2000.0
TH2_to_D2             2000.0
supply_electricity    2000.0
supply_geothermal     2000.0
Name: flow_cap_max, dtype: float64

In [405]:
model.inputs.sink_use_equals.sum(
    "timesteps", min_count=1, skipna=True
).to_series().dropna()

nodes  techs             
D1     demand_heat           500.0
D2     demand_heat           500.0
SH1    demand_electricity    300.0
TE1    demand_electricity      0.0
TE2    demand_electricity      0.0
TH1    demand_heat             0.0
TH2    demand_heat             0.0
Name: sink_use_equals, dtype: float64

In [406]:
model.build()
model.solve()

[2025-11-10 11:39:54] INFO     Model: backend build starting
[2025-11-10 11:39:55] INFO     Optimisation Model | parameters/lookups | Generated.
[2025-11-10 11:39:55] INFO     Optimisation Model | variables | Generated.
[2025-11-10 11:39:56] INFO     Optimisation Model | global_expressions | Generated.
[2025-11-10 11:39:57] INFO     Optimisation Model | constraints | Generated.
[2025-11-10 11:39:57] INFO     Optimisation Model | piecewise_constraints | Generated.
[2025-11-10 11:39:57] INFO     Optimisation Model | objectives | Generated.
[2025-11-10 11:39:57] INFO     Model: backend build complete
[2025-11-10 11:39:57] INFO     Optimisation model | starting model in base mode.
[2025-11-10 11:39:57] DEBUG    Set parameter Username
Set parameter LicenseID to value 2716243
Academic license - for non-commercial use only - expires 2026-09-30
Read LP format model from file C:\Users\alexn\AppData\Local\Temp\tmpy1nx13gr.pyomo.lp
Reading time = 0.00 seconds
x1: 96 rows, 107 columns, 246 nonzero

In [407]:
model.results

<xarray.Dataset> Size: 17kB
Dimensions:                     (nodes: 8, techs: 13, carriers: 2,
                                 timesteps: 1, costs: 1)
Coordinates:
  * techs                       (techs) object 104B 'SE1_to_TE2' ... 'supply_...
  * nodes                       (nodes) object 64B 'D1' 'D2' ... 'TH1' 'TH2'
  * carriers                    (carriers) object 16B 'electricity' 'heat'
  * timesteps                   (timesteps) datetime64[ns] 8B 2050-01-01
  * costs                       (costs) object 8B 'monetary'
Data variables: (12/20)
    flow_cap                    (nodes, techs, carriers) float64 2kB nan ... nan
    link_flow_cap               (techs) float64 104B 303.0 302.1 ... nan nan
    flow_out                    (nodes, techs, carriers, timesteps) float64 2kB ...
    flow_in                     (nodes, techs, carriers, timesteps) float64 2kB ...
    source_use                  (nodes, techs, timesteps) float64 832B nan .....
    source_cap                  (nodes, techs) float64 832B nan nan ... nan nan
    ...                          ...
    min_cost_optimisation       float64 8B 166.9
    capacity_factor             (nodes, techs, carriers, timesteps) float64 2kB ...
    systemwide_capacity_factor  (techs, carriers) float64 208B 0.4993 ... 1.0
    systemwide_levelised_cost   (techs, costs, carriers) float64 208B 0.00074...
    total_levelised_cost        (costs, carriers) float64 16B 0.5509 0.163
    unmet_sum                   (nodes, carriers, timesteps) float64 128B 0.0...

In [408]:
df_heat = (
    model.results.flow_out.sel(carriers="heat")
    .sum("nodes", min_count=1, skipna=True)
    .to_series()
    .dropna()
    .unstack("techs")
)

df_heat.head()

techs,SH1_to_TH1,TH1_to_D1,TH1_to_TH2,TH2_to_D2,supply_geothermal
timesteps,,,,,
2050-01-01,1010.060626,500.0,503.871821,500.0,1024.143077


In [409]:
df_electricity = (
    model.results.flow_out.sel(carriers="electricity")
    .sum("nodes", min_count=1, skipna=True)
    .to_series()
    .dropna()
    .unstack("techs")
)

df_electricity.head()

techs,SE1_to_TE2,SH1_to_TE1,TE1_to_D1,TE1_to_TE2,TE2_to_D2,supply_electricity
timesteps,,,,,,
2050-01-01,302.610673,300.0,0.0,302.073526,0.0,303.008613


In [410]:
costs = model.results.cost.to_series().dropna()
costs.head()

nodes  techs       costs   
D1     TE1_to_D1   monetary    0.000000
       TH1_to_D1   monetary    1.244608
D2     TE2_to_D2   monetary    0.000000
       TH2_to_D2   monetary    1.098110
SE1    SE1_to_TE2  monetary    0.113073
Name: cost, dtype: float64

In [411]:
# We set the color mapping to use in all our plots by extracting the colors defined in the technology definitions of our model.
colors = model.inputs.color.to_series().to_dict()

df_electricity = (
    (model.results.flow_out.fillna(0) - model.results.flow_in.fillna(0))
    .sel(carriers="electricity")
    .sum("nodes")
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow in/out (kWh)")
    .reset_index()
)
df_electricity_demand = df_electricity[df_electricity.techs == "demand_electricity"]
df_electricity_other = df_electricity[df_electricity.techs != "demand_electricity"]

print(df_electricity.head())

fig1 = px.bar(
    df_electricity_other,
    x="timesteps",
    y="Flow in/out (kWh)",
    color="techs",
    color_discrete_map=colors,
)
fig1.add_scatter(
    x=df_electricity_demand.timesteps,
    y=-1 * df_electricity_demand["Flow in/out (kWh)"],
    marker_color="black",
    name="demand",
)

                techs  timesteps  Flow in/out (kWh)
0          SE1_to_TE2 2050-01-01          -0.397940
1          SH1_to_TE1 2050-01-01          -2.073526
2          TE1_to_TE2 2050-01-01          -0.537146
3  demand_electricity 2050-01-01        -300.000000
4  supply_electricity 2050-01-01         303.008613


In [412]:
carriers = ["heat", "electricity"]
df_flows = (
    (model.results.flow_out.fillna(0) - model.results.flow_in.fillna(0))
    .sel(carriers=carriers)
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow in/out (kWh)")
    .reset_index()
)
df_demand = df_flows[df_flows.techs.str.contains("demand")]
df_flows_other = df_flows[~df_flows.techs.str.contains("demand")]

print(df_flows.head())

node_order = df_flows_other.nodes.unique()

fig = px.bar(
    df_flows_other,
    x="timesteps",
    y="Flow in/out (kWh)",
    facet_row="nodes",
    facet_col="carriers",
    color="techs",
    category_orders={"nodes": node_order, "carriers": carriers},
    height=1000,
    color_discrete_map=colors,
)

showlegend = True
# we reverse the node order (`[::-1]`) because the rows are numbered from bottom to top.
for row, node in enumerate(node_order[::-1]):
    for col, carrier in enumerate(carriers):
        demand_ = df_demand.loc[
            (df_demand.nodes == node) & (df_demand.techs == f"demand_{carrier}"),
            "Flow in/out (kWh)",
        ]
        if not demand_.empty:
            fig.add_scatter(
                x=model.results.timesteps.values,
                y=-1 * demand_,
                row=row + 1,
                col=col + 1,
                marker_color="black",
                name="Demand",
                legendgroup="demand",
                showlegend=showlegend,
            )
            showlegend = False
fig.update_yaxes(matches=None)
fig.show()

  nodes        techs     carriers  timesteps  Flow in/out (kWh)
0    D1    TH1_to_D1         heat 2050-01-01         500.000000
1    D1  demand_heat         heat 2050-01-01        -500.000000
2    D2    TH2_to_D2         heat 2050-01-01         500.000000
3    D2  demand_heat         heat 2050-01-01        -500.000000
4   SE1   SE1_to_TE2  electricity 2050-01-01        -303.008613


In [413]:
df_capacity = (
    model.results.flow_cap.where(
        ~model.inputs.base_tech.str.contains("demand|transmission")
    )
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow capacity (kW)")
    .reset_index()
)

print(df_capacity.head())

fig = px.bar(
    df_capacity,
    x="nodes",
    y="Flow capacity (kW)",
    color="techs",
    facet_col="carriers",
    color_discrete_map=colors,
)
fig.show()

  nodes               techs     carriers  Flow capacity (kW)
0   SE1  supply_electricity  electricity          303.008613
1   SH1   supply_geothermal         heat         1024.143077


In [414]:
df_coords = model.inputs[["latitude", "longitude"]].to_dataframe().reset_index()
df_capacity = (
    model.results.flow_cap.where(model.inputs.base_tech == "transmission")
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow capacity (kW)")
    .reset_index()
)
df_capacity_coords = pd.merge(df_coords, df_capacity, left_on="nodes", right_on="nodes").sort_values(by=['techs'])
fig1 = px.line_map(
    df_capacity_coords,
    lat="latitude",
    lon="longitude",
    color="carriers",
    hover_name="nodes",
    hover_data="Flow capacity (kW)",
    zoom=3,
    height=2000,
)
fig2 = px.scatter_map(
    df_capacity_coords,
    lat="latitude",
    lon="longitude",
    color="carriers",
    hover_name="nodes",
    hover_data="Flow capacity (kW)",
    zoom=3,
    height=2000,
)
fig=go.Figure(data = fig1.data + fig2.data)
fig.update_layout(
    map_style="open-street-map",
    map_zoom=16,
    map_center_lat=df_coords.latitude.mean(),
    map_center_lon=df_coords.longitude.mean(),
    margin={"r": 0, "t": 0, "l": 0, "b": 0},
    hoverdistance=50,
)